# Create American Academy in Rome Rome Prize Awards

Creates OpenAlex award rows from the official American Academy in Rome Rome Prize Fellows directory.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/aar_rome_prize_to_s3.py` to fetch the official directory and profile pages, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data source:** `https://www.aarome.org/people/rome-prize-fellows?field_affil_year_target_id=All` plus linked official fellow profile pages.  
**S3 location:** `s3a://openalex-ingest/awards/aar_rome_prize/aar_rome_prize_fellows.parquet`  
**Local full-source validation on 2026-05-28:** 612 Rome Prize fellow profile rows from official Drupal static HTML, grouped by 13 source disciplines/program headings, source years 2007-2026. Native IDs are unique; 100% profession/date/landing-url coverage; 99.8% project-title coverage; 99.3% project-description coverage; 99.7% named-fellowship coverage; 14 rows have two named recipients.

**OpenAlex funder mapping:**
- funder_id: 4320320895
- display_name: `American Academy in Rome`
- OpenAlex id: `https://openalex.org/F4320320895`
- ROR: NULL in OpenAlex public API as of 2026-05-28
- DOI: NULL in OpenAlex public API as of 2026-05-28

**Schema choices:**
- One row per official fellow profile, not one row per person. Joint profiles are represented with the first listed person as `lead_investigator` and the second listed person as `co_lead_investigator`.
- `display_name` prefers the source project title; it falls back to `Rome Prize {source_year} - {recipient_name}` for the one profile without a project title.
- `description` is the source project description, with the named fellowship preserved in raw staging and `funder_scheme`.
- `funder_scheme` = the exact named Rome Prize/fellowship when the profile publishes it, else the source discipline/program heading.
- `start_date` / `end_date` come from the profile fellowship date range. `source_year` is preserved separately in raw staging because the AAR class/source year usually corresponds to the fellowship end year.
- `amount` / `currency` are NULL by source authority and section 6.7 waiver: the public directory and profile pages do not publish per-fellow award amounts.


## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.aar_rome_prize_raw
USING delta
AS
SELECT
    *,
    current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/aar_rome_prize/aar_rome_prize_fellows.parquet`;

In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-28 found 612 rows.
SELECT COUNT(*) AS total_aar_rome_prize_rows
FROM openalex.awards.aar_rome_prize_raw;

In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.aar_rome_prize_raw;

In [ ]:
%sql
-- Sample raw AAR data.
SELECT
    funder_award_id,
    source_year,
    source_discipline,
    recipient_name,
    fellowship_name,
    profession,
    project_title,
    start_date,
    end_date,
    landing_page_url
FROM openalex.awards.aar_rome_prize_raw
LIMIT 10;

## Step 1.5: Inspect Raw Data, Dates, Native Keys, and Money Fields

In [ ]:
%sql
-- Money/currency field scan per runbook Step 1.5.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'aar_rome_prize_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant|award|currenc|ccy|usd|eur|gbp';

In [ ]:
%sql
-- Native key uniqueness, date coverage, and core field coverage.
SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT funder_award_id) AS distinct_native_ids,
    COUNT(display_name) AS has_display_name,
    COUNT(project_title) AS has_project_title,
    COUNT(project_description) AS has_project_description,
    COUNT(fellowship_name) AS has_fellowship_name,
    COUNT(profession) AS has_profession,
    COUNT(start_date) AS has_start_date,
    COUNT(end_date) AS has_end_date,
    ROUND(try_divide(COUNT(project_title), COUNT(*)) * 100.0, 1) AS pct_project_title,
    ROUND(try_divide(COUNT(project_description), COUNT(*)) * 100.0, 1) AS pct_project_description,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) AS pct_start_date,
    MIN(TRY_CAST(source_year AS INT)) AS min_source_year,
    MAX(TRY_CAST(source_year AS INT)) AS max_source_year
FROM openalex.awards.aar_rome_prize_raw;

In [ ]:
%sql
-- Source discipline/program-heading distribution from the official listing.
SELECT source_discipline, COUNT(*) AS rows
FROM openalex.awards.aar_rome_prize_raw
GROUP BY source_discipline
ORDER BY rows DESC, source_discipline;

In [ ]:
%sql
-- Named fellowship distribution. Most named prizes are endowed variants of the Rome Prize.
SELECT COALESCE(fellowship_name, source_discipline) AS scheme, COUNT(*) AS rows
FROM openalex.awards.aar_rome_prize_raw
GROUP BY COALESCE(fellowship_name, source_discipline)
ORDER BY rows DESC, scheme
LIMIT 25;

In [ ]:
%sql
-- Amount coverage is expected to be zero: the official directory does not publish per-fellow amounts.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies
FROM openalex.awards.aar_rome_prize_raw;

In [ ]:
%sql
-- Multi-recipient profile rows are represented as lead + co-lead in Step 2.
SELECT recipient_name, source_year, source_discipline, fellowship_name, landing_page_url
FROM openalex.awards.aar_rome_prize_raw
WHERE TRY_CAST(recipient_count AS INT) > 1
ORDER BY source_year, recipient_name;

## Step 1.6: Fail-fast - verify American Academy in Rome funder row exists

The Step 2 transform cross joins against `openalex.common.funder`. If this funder row is absent, the transform would silently emit zero awards. This assertion must pass before Step 2 runs.

In [ ]:
%sql
SELECT assert_true(
    COUNT(*) = 1,
    'Expected exactly one openalex.common.funder row for American Academy in Rome F4320320895'
) AS funder_row_exists
FROM openalex.common.funder
WHERE funder_id = 4320320895;

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320320895;

## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.aar_rome_prize_awards
USING delta
AS
WITH
aar_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320320895
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        NULLIF(TRIM(recipient_name), '') AS recipient_name_norm,
        NULLIF(TRIM(lead_given_name), '') AS lead_given_name_norm,
        NULLIF(TRIM(lead_family_name), '') AS lead_family_name_norm,
        NULLIF(TRIM(co_lead_given_name), '') AS co_lead_given_name_norm,
        NULLIF(TRIM(co_lead_family_name), '') AS co_lead_family_name_norm,
        NULLIF(TRIM(profession), '') AS profession_norm,
        COALESCE(NULLIF(TRIM(fellowship_name), ''), NULLIF(TRIM(source_discipline), '')) AS scheme_norm,
        NULLIF(TRIM(project_title), '') AS project_title_norm,
        NULLIF(TRIM(description), '') AS description_norm,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS parsed_end_date,
        TRY_CAST(source_year AS INT) AS parsed_source_year
    FROM openalex.awards.aar_rome_prize_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND recipient_name IS NOT NULL
      AND TRIM(recipient_name) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 AS id,

        COALESCE(
            g.project_title_norm,
            CONCAT('Rome Prize ', TRY_CAST(g.parsed_source_year AS STRING), ' - ', g.recipient_name_norm)
        ) AS display_name,

        g.description_norm AS description,

        f.funder_id,
        g.native_award_id AS funder_award_id,

        CAST(NULL AS DOUBLE) AS amount,
        CAST(NULL AS STRING) AS currency,

        struct(
            CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
            f.display_name,
            f.ror_id,
            f.doi
        ) AS funder,

        'fellowship' AS funding_type,

        COALESCE(g.scheme_norm, 'Rome Prize') AS funder_scheme,

        'aar_rome_prize_fellows' AS provenance,

        g.parsed_start_date AS start_date,
        g.parsed_end_date AS end_date,

        CASE
            WHEN COALESCE(YEAR(g.parsed_start_date), g.parsed_source_year) > YEAR(current_date()) + 1 THEN NULL
            ELSE COALESCE(YEAR(g.parsed_start_date), g.parsed_source_year)
        END AS start_year,
        CASE
            WHEN COALESCE(YEAR(g.parsed_start_date), g.parsed_source_year) > YEAR(current_date()) + 1 THEN NULL
            ELSE COALESCE(YEAR(g.parsed_end_date), g.parsed_source_year)
        END AS end_year,

        struct(
            g.lead_given_name_norm AS given_name,
            g.lead_family_name_norm AS family_name,
            CAST(NULL AS STRING) AS orcid,
            g.parsed_start_date AS role_start,
            struct(
                g.profession_norm AS name,
                CAST(NULL AS STRING) AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        ) AS lead_investigator,

        CASE
            WHEN g.co_lead_given_name_norm IS NULL AND g.co_lead_family_name_norm IS NULL THEN CAST(NULL AS STRUCT<
                given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
                affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
            >)
            ELSE struct(
                g.co_lead_given_name_norm AS given_name,
                g.co_lead_family_name_norm AS family_name,
                CAST(NULL AS STRING) AS orcid,
                g.parsed_start_date AS role_start,
                struct(
                    g.profession_norm AS name,
                    CAST(NULL AS STRING) AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                ) AS affiliation
            )
        END AS co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) AS investigators,

        NULLIF(TRIM(g.landing_page_url), '') AS landing_page_url,
        CAST(NULL AS STRING) AS doi,

        CONCAT('https://api.openalex.org/works?filter=awards.id:G',
               abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) AS works_api_url,

        current_timestamp() AS created_date,
        current_timestamp() AS updated_date

    FROM raw_prepared g
    CROSS JOIN aar_funder f
)

SELECT * FROM awards_transformed;

## Step 3: Insert into `openalex_awards_raw` at Priority 172

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'aar_rome_prize_fellows' AND priority = 172;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    172 AS priority  -- American Academy in Rome Rome Prize
FROM openalex.awards.aar_rome_prize_awards;

## Step 6: Verification

Amount/currency coverage is intentionally waived under section 6.7 because AAR does not publish per-fellow award amounts in the official directory or profile pages. The source does publish dates, named fellowships, professions/affiliations, project titles, and project descriptions; those should show high coverage.

In [ ]:
%sql
SELECT COUNT(*) AS total_aar_rome_prize_awards
FROM openalex.awards.aar_rome_prize_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.aar_rome_prize_awards;

In [ ]:
%sql
-- Data completeness, overall corpus.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(end_date) AS has_end_date,
    COUNT(lead_investigator) AS has_lead,
    COUNT(co_lead_investigator) AS has_co_lead,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) AS pct_start_date,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount
FROM openalex.awards.aar_rome_prize_awards;

In [ ]:
%sql
-- Amount/currency check. Expected: pct_amount = 0, currencies empty, waiver documented above.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    AVG(amount) AS avg_amount
FROM openalex.awards.aar_rome_prize_awards;

In [ ]:
%sql
-- Year distribution. Start years are fellowship start years; source_year is preserved in raw staging.
SELECT start_year, COUNT(*) AS rows
FROM openalex.awards.aar_rome_prize_awards
GROUP BY start_year
ORDER BY start_year;

In [ ]:
%sql
-- Future-year cap check: must be zero.
SELECT COUNT(*) AS future_start_year_rows
FROM openalex.awards.aar_rome_prize_awards
WHERE start_year > YEAR(current_date()) + 1;

In [ ]:
%sql
-- Funder struct sanity: all rows should point to American Academy in Rome.
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.aar_rome_prize_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Scheme distribution after fallback from named fellowship to source discipline.
SELECT funder_scheme, COUNT(*) AS rows
FROM openalex.awards.aar_rome_prize_awards
GROUP BY funder_scheme
ORDER BY rows DESC, funder_scheme
LIMIT 25;

In [ ]:
%sql
-- Confirm joint profile rows populate co_lead_investigator.
SELECT
    funder_award_id,
    start_year,
    lead_investigator.given_name AS lead_given,
    lead_investigator.family_name AS lead_family,
    co_lead_investigator.given_name AS co_given,
    co_lead_investigator.family_name AS co_family,
    landing_page_url
FROM openalex.awards.aar_rome_prize_awards
WHERE co_lead_investigator IS NOT NULL
ORDER BY start_year, funder_award_id;

In [ ]:
%sql
-- Sample 10 rows for human QA.
SELECT
    id,
    SUBSTRING(display_name, 1, 90) AS display_name,
    SUBSTRING(description, 1, 120) AS description,
    funder_scheme,
    start_date,
    end_date,
    start_year,
    lead_investigator.given_name AS given_name,
    lead_investigator.family_name AS family_name,
    SUBSTRING(lead_investigator.affiliation.name, 1, 70) AS profession_or_affiliation,
    landing_page_url
FROM openalex.awards.aar_rome_prize_awards
ORDER BY start_year DESC, family_name
LIMIT 10;

## Handoff Notes

Contractor validation completed locally on 2026-05-28:
- `python3 -m py_compile scripts/local/aar_rome_prize_to_s3.py`
- `python3 scripts/local/aar_rome_prize_to_s3.py --limit 10 --output-dir /tmp/aar_rome_prize_smoke --skip-upload`
- `python3 scripts/local/aar_rome_prize_to_s3.py --output-dir /tmp/aar_rome_prize_full --skip-upload`
- `python3 scripts/local/aar_rome_prize_to_s3.py --output-dir /tmp/aar_rome_prize_full --skip-download --skip-upload`

Expected full row count: 612. Admin must upload the parquet, run this notebook in Databricks, inspect the verification cells, and only then decide whether to add any job wiring.
